In [1]:
import pandas as pd
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [2]:
print(train.head())
 

            ID                                               Text  \
0  train_00001  Its "duet" feature allows users to film a vide...   
1  train_00002  *@**%%@ To support this, Blizzard released the...   
2  train_00003  James Mitchell, the Premier of Western Austral...   
3  train_00004  Pharo has an implementation of a heap in the C...   
4  train_00005  a for Tour, Alberto his his possible allowed m...   

             Subject  
0        Pop Culture  
1             Gaming  
2            History  
3  Computer Sciences  
4             Sports  


In [3]:
print(train.shape)

(10000, 3)


In [4]:

print(test.head())

          ID                                               Text
0  test_0001  Square's decision to produce games %*@*%@@ exc...
1  test_0002  Many of the properties in the Phase are set af...
2  test_0003  As of at least 2015, Apple has removed legacy ...
3  test_0004  Roman coins and medieval artefacts have all be...
4  test_0005   Thor: Love and Thunder is also set after Endgame


Preprocessing the text

In [5]:
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords   

stopwords = set(stopwords.words('english'))


[nltk_data] Downloading package stopwords to C:\Users\Shubhali
[nltk_data]     Chaudhary\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation and special characters
    text = re.sub(r'[^a-z\s]', '', text)
    # Remove stopwords
    text = ' '.join(word for word in text.split() if word not in stopwords)
    return text

In [7]:
train['cleaned_text'] = train['Text'].apply(preprocess_text)
test['cleaned_text'] = test['Text'].apply(preprocess_text)

In [8]:
print(train.head())

            ID                                               Text  \
0  train_00001  Its "duet" feature allows users to film a vide...   
1  train_00002  *@**%%@ To support this, Blizzard released the...   
2  train_00003  James Mitchell, the Premier of Western Austral...   
3  train_00004  Pharo has an implementation of a heap in the C...   
4  train_00005  a for Tour, Alberto his his possible allowed m...   

             Subject                                       cleaned_text  
0        Pop Culture  duet feature allows users film video aside ano...  
1             Gaming  support blizzard released hero reference kit r...  
2            History  james mitchell premier western australia lent ...  
3  Computer Sciences  pharo implementation heap collectionssequencea...  
4             Sports  tour alberto possible allowed mistakefree star...  


Vectorization using tf-IDF

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=5000,ngram_range=(1, 3))
x_train = vectorizer.fit_transform(train['cleaned_text'])
x_test = vectorizer.transform(test['cleaned_text'])

In [10]:
y_train=train['Subject']

Model making and training

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
#split for validation
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=1000, random_state=42)
model.fit(x_train, y_train)

,n_estimators,1000
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [12]:
y_pred = model.predict(x_val)

In [13]:
from sklearn.metrics import classification_report, f1_score
print(classification_report(y_val, y_pred)) 
print("F1 Score:", f1_score(y_val, y_pred, average='macro'))
print("Score", 100 * f1_score(y_val, y_pred, average='macro'))

                   precision    recall  f1-score   support

Computer Sciences       0.85      0.84      0.84       209
           Gaming       0.79      0.75      0.77       322
        Geography       0.79      0.80      0.79       283
          History       0.87      0.75      0.81       174
 Natural Sciences       0.80      0.81      0.81       273
      Pop Culture       0.73      0.75      0.74       315
           Sports       0.77      0.83      0.80       424

         accuracy                           0.79      2000
        macro avg       0.80      0.79      0.79      2000
     weighted avg       0.79      0.79      0.79      2000

F1 Score: 0.7935099689358412
Score 79.35099689358412


Prediction for submission

In [14]:
test_preds = model.predict(x_test)
submission = pd.DataFrame({'ID': test['ID'], 'Subject': test_preds})        


In [15]:
print(submission.head())

          ID            Subject
0  test_0001             Gaming
1  test_0002   Natural Sciences
2  test_0003  Computer Sciences
3  test_0004            History
4  test_0005        Pop Culture


In [16]:
submission.to_csv('submission.csv', index=False)

XGBOOST MODEL

In [17]:
import xgboost as xgb

In [18]:
from sklearn.preprocessing import LabelEncoder

label_encoder=LabelEncoder()
train['Subject'] = label_encoder.fit_transform(train['Subject'])

In [19]:
y__train = train['Subject']
print(y__train.shape)
print(y__train.unique())

(10000,)
[5 1 3 0 6 4 2]


In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=5000,ngram_range=(1, 3))
x_train = vectorizer.fit_transform(train['cleaned_text'])
x_test = vectorizer.transform(test['cleaned_text'])

In [21]:
#split for validation
x_tr, x_val, y_tr, y_val = train_test_split(x_train, y__train, test_size=0.2, random_state=42)

In [22]:
xgb_model = xgb.XGBClassifier(eval_metrics='mlogloss',n_estimators=1000, random_state=42,use_label_encoder=False,max_depth=6,learning_rate=0.1,subsample=0.8,colsample_bytree=0.8)

In [23]:
xgb_model.fit(X=x_tr, y=y_tr,eval_set=[(x_val, y_val)],  verbose=True)

d:\fractal_challenge\.venv\lib\site-packages\xgboost\training.py:183: UserWarning: [11:49:58] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "eval_metrics", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[0]	validation_0-mlogloss:1.91464
[1]	validation_0-mlogloss:1.88901
[2]	validation_0-mlogloss:1.86924
[3]	validation_0-mlogloss:1.85113
[4]	validation_0-mlogloss:1.83486
[5]	validation_0-mlogloss:1.82172
[6]	validation_0-mlogloss:1.80765
[7]	validation_0-mlogloss:1.79670
[8]	validation_0-mlogloss:1.78643
[9]	validation_0-mlogloss:1.77717
[10]	validation_0-mlogloss:1.76912
[11]	validation_0-mlogloss:1.75983
[12]	validation_0-mlogloss:1.75250
[13]	validation_0-mlogloss:1.74482
[14]	validation_0-mlogloss:1.73798
[15]	validation_0-mlogloss:1.73111
[16]	validation_0-mlogloss:1.72393
[17]	validation_0-mlogloss:1.71840
[18]	validation_0-mlogloss:1.71289
[19]	validation_0-mlogloss:1.70750
[20]	validation_0-mlogloss:1.70256
[21]	validation_0-mlogloss:1.69764
[22]	validation_0-mlogloss:1.69250
[23]	validation_0-mlogloss:1.68935
[24]	validation_0-mlogloss:1.68598
[25]	validation_0-mlogloss:1.68268
[26]	validation_0-mlogloss:1.67902
[27]	validation_0-mlogloss:1.67527
[28]	validation_0-mlogloss:1.6

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [24]:
y_pred = xgb_model.predict(x_val)

In [25]:
from sklearn.metrics import classification_report,f1_score
f1_score=f1_score(y_val, y_pred, average='macro')

In [26]:
score=100* f1_score
print("F1 Score:", f1_score)

F1 Score: 0.7945818282784273


In [27]:
classification_report= classification_report(y_val, y_pred)
print(classification_report)

              precision    recall  f1-score   support

           0       0.83      0.84      0.83       209
           1       0.81      0.77      0.79       322
           2       0.79      0.80      0.79       283
           3       0.87      0.74      0.80       174
           4       0.79      0.79      0.79       273
           5       0.78      0.73      0.75       315
           6       0.75      0.85      0.80       424

    accuracy                           0.79      2000
   macro avg       0.80      0.79      0.79      2000
weighted avg       0.79      0.79      0.79      2000



In [28]:
x_test = vectorizer.transform(test['Text'])

In [29]:
y_pred_encoded = xgb_model.predict(x_test)
y_pred_labels = label_encoder.inverse_transform(y_pred_encoded)

In [30]:
output_df = pd.DataFrame({'ID': test['ID'], 'Subject': y_pred_labels})
print(output_df.head())
output_df.to_csv('submission.csv', index=False)

          ID            Subject
0  test_0001             Gaming
1  test_0002        Pop Culture
2  test_0003  Computer Sciences
3  test_0004            History
4  test_0005        Pop Culture


In [32]:
!pip install joblib

Defaulting to user installation because normal site-packages is not writeable


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: C:\Program Files\Python310\python.exe -m pip install --upgrade pip


In [34]:
import joblib
joblib.dump(xgb_model, 'xgb_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')
joblib.dump(label_encoder, 'label_encoder.pkl')

['label_encoder.pkl']